In [55]:
from langchain_core.runnables import Runnable
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

class FetchYoutubeTranscript(Runnable):

    def invoke(self, id: str, config=None, **kwargs):
        api = YouTubeTranscriptApi()
        transcript = api.fetch(
            video_id=id,
            languages=("en",),
            preserve_formatting=False
        )
        return transcript
    
class MergeTranscript(Runnable):

    def invoke(self, transcript, config=None, **kwargs):
        return " ".join([t.text for t in transcript])
    
class CreateTranscriptChunks(Runnable):

    def __init__(self, chunk_size=500, chunk_overlap=100):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )
    def invoke(self, text: str, config=None, **kwargs):
        return self.splitter.split_text(text)
    
class ConvertChunksIntoDocument(Runnable):
    
    def invoke(self, chunks, config=None, **kwargs):
        docs = [Document(page_content=chunk) for chunk in chunks]
        return docs
    
class StoreEmbeddingsIntoVectorStore(Runnable):

    def __init__(self):
        self.embedding_model = HuggingFaceEndpointEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

    def invoke(self, docs, config=None, **kwargs):
        vectore_store = FAISS.from_documents(
            documents=docs,
            embedding=self.embedding_model
        )
        return vectore_store
    
class GetUserQuery(Runnable):

    def invoke(self, query, config=None, **kwargs):
        return str(query)
    
class RetrieveSimilarDocuments(Runnable):

    def __init__(self, vector_store):
        self.vector_store = vector_store

    def invoke(self, query, config=None, **kwargs):
        retriever = self.vector_store.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 1}
        )
        retrieved_docs = retriever.invoke(query)
        return {
            "retrieved_document": retrieved_docs,
            "query": query,
        }

class GeneratePrompt(Runnable):

    def __init__(self):
        self.template = PromptTemplate(template="""
            You are an expert assistant specialized in analyzing YouTube video content. 
            Your goal is to provide accurate, concise, and helpful answers based ONLY on the transcript provided below.

            ---
            TRANSCRIPT CONTEXT:
                {context}
            ---

            USER QUESTION: 
                {question}

            INSTRUCTIONS:
            1. Use the provided transcript context to answer the question.
            2. If the answer is not contained within the context, honestly state that the video does not cover this topic. 
             3. Do not use outside knowledge or hallucinate facts.
            4. Keep your tone professional and objective.
            """,
            input_variables=["context", "question"]
        )

    def invoke(self, inputs, config=None, **kwargs):
        docs = inputs["retrieved_document"]
        query = inputs["query"]

        context = " ".join([doc.page_content for doc in docs])

        prompt = self.template.format(
            context=context,
            question=query
        )
        return prompt
    
class HuggingFaceLLM(Runnable):

    def __init__(self):
        self.llm = HuggingFaceEndpoint(
            repo_id="mistralai/Mistral-7B-Instruct-v0.2",
            task="text-generation"
        )

        self.model = ChatHuggingFace(llm=self.llm)

    def invoke(self, query, config=None, **kwargs):
        response = self.model.invoke(query)
        return response.content

class OutputParser(Runnable):

    def __init__(self):
        self.parser = StrOutputParser()

    def invoke(self, response, config=None, **kwargs):
        parsed_output = self.parser.parse(response)
        return parsed_output

In [56]:
fetcher = FetchYoutubeTranscript()
merger = MergeTranscript()
chunker = CreateTranscriptChunks()
document_converter = ConvertChunksIntoDocument()
vector_store = StoreEmbeddingsIntoVectorStore()

vector_store_chain = fetcher | merger | chunker | document_converter | vector_store

vector_store = vector_store_chain.invoke("67_aMPDk2zw")

query_input = input("Enter query: ")

query_runnable = GetUserQuery()
retriever = RetrieveSimilarDocuments(vector_store)
prompt_template = GeneratePrompt()
llm = HuggingFaceLLM()
parser = OutputParser()

chain = query_runnable | retriever | prompt_template | llm | parser

result = chain.invoke(query_input)

In [ ]:
print(result)

LLM, or Large Language Model, is a type of artificial intelligence model that is designed to process and understand human language.

The transcript context discusses various applications and approaches related to LLMs. For example, it mentions ChatGPT, which uses an LLM called GPT-3 or GPT-4 behind the scenes.

Additionally, the context discusses various other applications and approaches related to LLMs, including Palm 2 by Google, LLAMA by Meta, and reinforcement learning with human feedback (RLHF).

Overall, the context provides a wealth of information about various applications and approaches related to LLMs. If you have any specific questions about these topics, please let me know and I will do my best to provide you with accurate and helpful information based on the context provided.
